In [14]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import text, create_engine, bindparam, Engine
import os
from dotenv import load_dotenv
import re
from datetime import datetime
load_dotenv()
OLD_DB = os.getenv("OLD_DB")
DB_HOST = os.getenv("DB_HOST")
engine = create_engine(DB_HOST)
old_engine = create_engine(OLD_DB)
SUBSTATION_ID = ["CIPC0001"]
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_columns = None
from template_generator.dsas.template_generator import TemplateGenerator
import calendar
BILLING_PERIOD = "202501"

<div class="alert alert-success">
 <h3>
 <strong>
    PRIMARYLINES
 </strong>
 </h3>
</div>

<div class="alert alert-success">
 <h3>
 <strong>
    DISTRIBUTION TRANSFORMER
 </strong>
 </h3>
</div>

<div class="alert alert-success">
 <h3>
 <strong>
    SECONDARY LINES
 </strong>
 </h3>
</div>

In [63]:
from template_generator.dsas.secondary_lines import SecondaryLines

secondary_lines = SecondaryLines(SUBSTATION_ID, BILLING_PERIOD, engine)
secondary_lines.save_secondary_lines()

AttributeError: 'SecondaryLines' object has no attribute 'process_secondary_lines'

In [54]:
consumer_meter = ConsumerMeter(SUBSTATION_ID, BILLING_PERIOD, engine)
bus = BusData(SUBSTATION_ID, BILLING_PERIOD, engine)
primary_lines = PrimaryLines(SUBSTATION_ID, BILLING_PERIOD, engine)
primary_lines.save_pred_rel_data()
distribution_transformer = DistributionTransformer(SUBSTATION_ID, BILLING_PERIOD, engine)
distribution_transformer.save_distribution_transformer_data()


,count,transformer_id,from_primary_bus_id,to_secondary_bus_id,primary_phasing,secondary_phasing,installation_type,no_of_dts_in_bank,connection_code,kva_rating,primary_voltage_rating,secondary_voltage_rating,primary_tap_voltage_kv,secondary_tap_voltage_kv,X/R Ratio,No-Load Loss (kW),Exciting Current (%)
0,1,STA0139,F1_P001A1001,OSTA0139,ABCN,ABCN,PAD-MOUNTED,4,4,15.00,7.96,0.24,7.96,0.24,1.50,0.01,0.09
1,2,DTA0001,F4,S000A0031,BN,BN,POLE-MOUNTED,1,1,10.00,7.96,0.24,7.96,0.24,1.50,0.01,0.06
2,3,STA0125,F2_P002A0054,OSTA0125,ABCN,ABCN,PAD-MOUNTED,4,4,100.00,7.96,0.24,7.96,0.24,1.50,0.05,0.63
3,4,STA0449,F1_P001A0424,OSTA0449,ABCN,ABCN,POLE-MOUNTED,4,10,167.00,7.96,0.24,7.96,0.24,NaN,NaN,NaN
4,5,STA0194,F1_P001A0125,OSTA0194,CN,CN,POLE-MOUNTED,1,1,25.00,7.96,0.24,7.96,0.24,1.50,0.02,0.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,513,DTB0093,F3_P003B1063,S003B1063,AN,AN,POLE-MOUNTED,1,1,15.00,7.96,0.24,7.96,0.24,1.50,0.01,0.09
513,514,DTB0096,F3_P003B1101,S003B1101,AN,AN,POLE-MOUNTED,1,1,10.00,7.96,0.24,7.96,0.24,1.50,0.01,0.06
514,515,DTB0094,F3_P003B1072,S003B1072,AN,AN,POLE-MOUNTED,1,1,10.00,7.96,0.24,7.96,0.24,1.50,0.01,0.06
515,516,DTB0089,F3_P003B1095,S003B1095,AN,AN,POLE-MOUNTED,1,1,25.00,7.96,0.24,7.96,0.24,1.50,0.02,0.16


#### CHECKING AND VALIDATION ON MONTHLY FINANCIAL SYSTEMLOSS REPORT (MFSR)

Uppon checking on the result of total consumer kwh_used for Coron and Busuanga Sometime the MFSR data is equal to the kwh_used total per consumer. So to scale down or up the kwh_used to match the mfsr data the solution are:
- Compare the different per each municipality mfsr kwh_sale vs per consumer total_kwh
- Then I merge the result different back to clean_kwh_sale and multiply the different into original kwh_sale per consumer to match the total kwh_sale in the mfsr.

<div class="alert alert-info">
<strong>Note:</strong> MFSR data is private.
</div>

<div class="alert alert-success">
<strong> Processing MFSR Data</strong>
</div>



In [138]:

YEAR=2025
mfsr_dir = "mfsr"
class MFSR:
    def __init__(self, folder, YEAR, db_engine:Engine):
        self.folder = folder
        self.clean_df = []
        self.YEAR = YEAR
        self.engine = db_engine
    # READ AND PROCESS
    def read_and_process(self):
        mfsr_file = os.listdir(self.folder)
        for file in [f for f in mfsr_file if re.search(str(self.YEAR), f)]:
            mfsr_sheet = 'Sched 1-GS'
            use_cols = 'b:N'
            skip_row = 9
            # FORMATING BILLING PERIOD
            ptime_format = "%B%Y"
            ftime_format = "%Y%m"
            billing_period = datetime.strptime(file.split("@")[1].split(".")[0], ptime_format).strftime(ftime_format)
            
            # READ AND PROCESS THE DATA
            mfsr_df = pd.read_excel(os.path.join(mfsr_dir, file), sheet_name=mfsr_sheet, skiprows=skip_row, usecols=use_cols)
            mfsr_df.rename(columns={
                mfsr_df.columns[0]: "CATEGORY",
            }, inplace=True)
        
            mfsr_df['CATEGORY'] = mfsr_df['CATEGORY'].str.replace(
                r"^\s*(?:\(\d+\)\s*[A-Za-z]?|[A-Za-z0-9]+\.)\s*", 
                "", 
                regex=True,
            )
            mfsr_df.dropna(subset=mfsr_df.columns[1:], inplace=True, how="all")
            mfsr_df.fillna(0, inplace=True)
            mfsr_df = mfsr_df.iloc[:5, :]
            for mfsr_col in mfsr_df.columns[1:]:
                mfsr_df[mfsr_col] = mfsr_df[mfsr_col].astype(float)
            mfsr_df['CORON_BUSUANGA'] = mfsr_df['CORON'] + mfsr_df['BUSUANGA']
            mfsr_df = mfsr_df.set_index("CATEGORY").T
            mfsr_df.columns.name = None
            mfsr_df.index.name = "FEEDER"
            mfsr_df = mfsr_df.reset_index()
            mfsr_df['billing_period'] = billing_period
            mfsr_df['billing_period'] = mfsr_df['billing_period'].astype("string")
            self.clean_df.append(mfsr_df)
        return self.clean_df
    # SAVE DATA
    def save_data(self,destination:str):
        try: 
            stmt = text("""
                        select sub.substation_id, mun.name as municipality, vill.name as village from gis.substation as sub
                        join gis.villages  as vill
                        on vill.id = sub.village_id
                        join gis.municipality as mun
                        on mun.id = sub.municipal_id;
                        """)
            data = pd.read_sql(stmt, self.engine)
            data = data[data['substation_id'] !='CIPC0002']
            data['FEEDER'] = data['village'].apply(lambda x: 'CORON_BUSUANGA' if re.search("GUADALUPE", x) else 'CULION' if x == 'OSMEÑA' else x)
            mfsr_df = pd.concat(self.clean_df)
            mfsr_df = mfsr_df.merge(data, how="left", left_on="FEEDER", right_on="FEEDER")
            mfsr_df = mfsr_df[~mfsr_df['FEEDER'].isin(['CORON', 'BUSUANGA'])]
            mfsr_df.to_csv(os.path.join(destination, f"mfsr_{self.YEAR}.csv"), index=False)
        except Exception as e:
            raise e
        return f"{destination} saved"

### SAVE THE PROCESS AND CLEAN MFSR DATA

In [139]:
# MFSR DATA
mfsr = MFSR(mfsr_dir, YEAR, engine)
mfsr.read_and_process()
destination_folder = os.path.join(mfsr_dir,"cleaned")
os.makedirs(destination_folder, exist_ok=True)
print(mfsr.save_data(destination_folder))

mfsr\cleaned saved


### UPDATE SECONDARY LINES

In [40]:

old_db_stmt = text(
    """select secondary_line_id, conductor_type from gis.secondary_line as sl"""
)
sl_df = pd.read_sql(old_db_stmt, old_engine)
conductor_type_stmt = text(
    """select id, name from gis.conductor_wires"""
)
ct_df = pd.read_sql(conductor_type_stmt, engine)

sl_df = sl_df.merge(ct_df, how="left", left_on="conductor_type", right_on="name")
sl_df = sl_df[['secondary_line_id', 'id']]
sl_df.rename(columns={
    'id' : 'conductor_type'
}, inplace=True)


update_stmt = text(
    """update gis.secondary_lines
    set conductor_type = :conductor_type
    where secondary_line_id = :secondary_line_id
    """
)
records = sl_df.to_dict(orient="records")
try:
    with engine.begin() as conn:
        conn.execute(update_stmt,parameters=records)
        conn.commit()
        print("complete")
except Exception as e: 
    print(e)

complete
